In [1]:
import pandas as pd
import re

In [2]:
dobrava = pd.read_csv('../data/raw/dobrava_belgrade_all.csv')
pd.set_option('display.max_colwidth', None)

In [3]:
dobrava.shape

(790, 28)

In [4]:
tmp = dobrava[dobrava['GenBank_Title'].str.contains('METHODS', case=False, na=False)]
display(tmp[['GenBank_Title']].head(10))

,GenBank_Title
509,JP 2011191322-A/4: METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
510,JP 2011191322-A/10: METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
511,JP 2011191322-A/16: METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
600,METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
601,METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
602,METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
692,METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
693,METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION
694,METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION


Uklonjeni su zapisi koji sadrze rec 'METHODS' jer oni nisu stvarne bioloske sekvence i ne predstavljaju S,M ili L sekvence.

In [5]:
trash_words = 'METHODS'
dobrava = dobrava[
    ~dobrava['GenBank_Title'].str.contains(trash_words, case=False, na=False)
].copy()

In [6]:
dobrava.shape

(781, 28)

In [7]:
is_nan_segment = dobrava['Segment'].isna()

In [8]:
dobrava['Segment'].value_counts(dropna = False)

Segment
M             208
S             205
L             191
NaN           168
medium (M)      5
N               2
small (S)       2
Name: count, dtype: int64

U sledećim zapisima S segment nije bio označen kao S, već kao N (nucleocapsid segment). Ovi zapisi su mapirani na S segment, jer S segment hantavirusa kodira nuklokapsidni protein.	

In [9]:
dobrava['Segment'] = dobrava['Segment'].replace({
    'N': 'S',
    'small (S)': 'S',
    'medium (M)': 'M',
    'Small': 'S',
    'Medium': 'M',
    'Large': 'L'
})

In [10]:
dobrava['Segment'].value_counts(dropna = False)

Segment
M      213
S      209
L      191
NaN    168
Name: count, dtype: int64

Za nepoznate vrednosti sekvence popunjavamo vrednosti na osnovu kolone GenBank_Title. S segment kodira nukleokapsidni protein, M segment kodira glikoproteine, dok L segment kodira RNK zavaisnu polimerazu.

In [11]:
dobrava.loc[ is_nan_segment &
    dobrava['GenBank_Title'].str.contains(
        'nucleocapsid|nucleoprotein',
        case=False,
        na=False
    ),
    'Segment'
] = 'S'

dobrava.loc[ is_nan_segment &
    dobrava['GenBank_Title'].str.contains(
        'glycoprotein|M polyprotein gene',
        case=False,
        na=False
    ),
    'Segment'
] = 'M'

dobrava.loc[ is_nan_segment &
    dobrava['GenBank_Title'].str.contains(
        'RNA-dependent RNA polymerase|L segment',
        case=False,
        na=False
    ),
    'Segment'
] = 'L'

In [12]:
dobrava['Segment'].value_counts(dropna=False)

Segment
S    323
M    239
L    219
Name: count, dtype: int64

In [13]:
cols = ['Accession','Segment',  'Nuc_Completeness', 'Species']

In [14]:
all_seq = dobrava[cols]
all_seq.to_csv('../data/processed_sequences/all_sequences/dobrava_all.csv')

In [15]:
complete_sequences = all_seq[all_seq['Nuc_Completeness'] == 'complete'].copy()
complete_sequences.to_csv('../data/processed_sequences/complete_sequences/dobrava_complete.csv')

In [16]:
dobrava[dobrava['Accession'] == 'KU561220.1']['GenBank_Title']

171    Orthohantavirus dobravaense strain DOBV/Bachorzec-Slonne/Af-51 segment M glycoprotein precursor, gene, partial cds
Name: GenBank_Title, dtype: object

In [17]:
dobrava.columns

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family', 'Genotype',
       'Isolate', 'Segment', 'GenBank_Title', 'Length', 'Nuc_Completeness',
       'Geo_Location', 'Country', 'USA', 'Host', 'Tissue_Specimen_Source',
       'Submitters', 'Organization', 'Org_location', 'Publications',
       'Collection_Date', 'Release_Date', 'Molecule_type'],
      dtype='object')